In [ ]:
TEST_PROMPT = """
你是一位资深的 Python 测试工程师。请为提供的代码生成一套完整的 pytest 测试用例。

要求：
1. 在适当的情况下使用 pytest fixture。
2. 测试正常路径（符合预期的常规输入）。
3. 测试边界情况（边界值、空输入等）。
4. 测试异常情况（非法输入、异常抛出等）。
5. 使用具有描述性的测试函数命名，格式为：`test_函数名_场景`。
6. 为每个测试添加简短的文档字符串（docstring）。
7. 对重复性的测试场景使用 `pytest.mark.parametrize`。
8. 对外部依赖（如 API 调用、文件 I/O、数据库等）进行 Mock。
9. 测试目标应达到 90% 以上的代码覆盖率。

输出要求：
仅输出完整的测试文件内容，可直接使用 `pytest` 运行，不要输出任何额外的解释或说明。
"""

In [ ]:
SAMPLE_CODE = '''
def calculate_discount(price: float, discount_percent: float) -> float:
    """Calculate discounted price."""
    if price < 0:
        raise ValueError("Price cannot be negative")
    if not 0 <= discount_percent <= 100:
        raise ValueError("Discount must be between 0 and 100")
    return price * (1 - discount_percent / 100)


def find_longest_word(text: str) -> str:
    """Find the longest word in a text string."""
    if not text or not text.strip():
        return ""
    words = text.split()
    return max(words, key=len)


class ShoppingCart:
    def __init__(self):
        self.items = {}

    def add_item(self, name: str, price: float, quantity: int = 1):
        if price < 0:
            raise ValueError("Price cannot be negative")
        if quantity < 1:
            raise ValueError("Quantity must be at least 1")
        if name in self.items:
            self.items[name]["quantity"] += quantity
        else:
            self.items[name] = {"price": price, "quantity": quantity}

    def remove_item(self, name: str):
        if name not in self.items:
            raise KeyError(f"Item '{name}' not in cart")
        del self.items[name]

    def total(self) -> float:
        return sum(item["price"] * item["quantity"] for item in self.items.values())
'''

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import os

load_dotenv()

model = os.getenv('MODEL_NAME', '')

llm = ChatOpenAI(model=model, temperature=0.3, extra_body={'enable_thinking': False})

def generate_test(code: str, filename: str = 'module') -> str:
    messages = [
        SystemMessage(content=TEST_PROMPT),
        HumanMessage(content=f"""请为 {filename} 中的 Python 代码生成一套完整的 pytest 测试用例：\n\n```python\n{code}\n```""")
    ]
    return llm.stream(messages)

In [8]:
for chunk in generate_test(code=SAMPLE_CODE):
    print(chunk.content, end='', flush=True)

```python
import pytest
from unittest.mock import patch, MagicMock


# Fixtures
@pytest.fixture
def shopping_cart():
    """Create a fresh ShoppingCart instance for each test."""
    from module import ShoppingCart
    return ShoppingCart()


# Tests for calculate_discount
class TestCalculateDiscount:
    
    def test_calculate_discount_normal_case(self):
        """Test normal discount calculation with valid inputs."""
        from module import calculate_discount
        result = calculate_discount(100.0, 10.0)
        assert result == 90.0

    def test_calculate_discount_zero_discount(self):
        """Test discount calculation with 0% discount."""
        from module import calculate_discount
        result = calculate_discount(100.0, 0.0)
        assert result == 100.0

    def test_calculate_discount_full_discount(self):
        """Test discount calculation with 100% discount."""
        from module import calculate_discount
        result = calculate_discount(100.0, 100.0)
   